In [1]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [2]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [21]:
import json
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [16]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [20]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

openai_client = OpenAI()
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [28]:
response.output[0].content[0].text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. If you’re just learning, you can start anytime using the course materials.\n\nIf you’d like, I can also help you figure out the best way to start or explain the certificate requirements. Any other areas you want to explore?'

In [ ]:
messages.extend(response.output)
for item in response.output:
    #print(item)

    if item.type == 'function_call':
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
    elif item.type == 'message':
        print("ASSISTANT:")
        print(item.content[0].text)

ResponseOutputMessage(id='msg_0e422c38cf12405d006a3695ae62448195aab225f3747c76f7', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. If you’re just learning, you can start anytime using the course materials.\n\nIf you’d like, I can also help you figure out the best way to start or explain the certificate requirements. Any other areas you want to explore?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. If you’re just learning, you can start anytime using the course materials.

If you’d like, I can also help you figure out the best way to start or explain the certificate requirements. Any other areas you want to explore?


In [18]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [32]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1
while True: 
    print(f'iteration# {it}')
    has_function_calls = False 
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)
    for item in response.output:
        #print(item)

        if item.type == 'function_call':
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True
        elif item.type == 'message':
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it  + 1
    if has_function_calls == False:
        break

iteration# 1
function_call: search {"query":"join course discovered can I join enrollment registration late join FAQ"}
iteration# 2
ASSISTANT:
Yes — you can still join the course. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.

If you'd like, I can also help with how to start the course or explain the certificate requirements. Are there other areas you want to explore?


In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find any course/FAQ entry about “queen gambit,” so I can’t answer that from the course materials.

If you meant something course-related, try rephrasing it with the specific topic or lecture name. Is there another area you want to explore?


'I couldn’t find any course/FAQ entry about “queen gambit,” so I can’t answer that from the course materials.\n\nIf you meant something course-related, try rephrasing it with the specific topic or lecture name. Is there another area you want to explore?'

In [35]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [36]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Ollama locally install local run Ollama"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 run llama3 course FAQ"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```

4. **Optional: use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If you get a 

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Optional: use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, you may need to restart the server with:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you how to run Ollama in Docker or 

In [38]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [39]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [40]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [45]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [46]:
chat_interface = IPythonChatInterface()

In [47]:
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [48]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [49]:
result.cost

CostInfo(input_cost=Decimal('0.00277575'), output_cost=Decimal('0.0013905'), total_cost=Decimal('0.00416625'))